# 🧠 MODMA 3-Channel EEG — Preprocessing Pipeline (Colab)

## 0 · Runtime check — confirm GPU

In [ ]:
import subprocess, sys

# Confirm GPU is available
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected:", result.stdout.strip())
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → GPU (T4 or A100)")

print("Python:", sys.version.split()[0])

## 1 · Install dependencies

In [ ]:
%%capture
!pip install PyWavelets mne scipy scikit-learn openpyxl pandas numpy matplotlib seaborn

print("✅ Packages ready")

## 2 · Imports

In [ ]:
import os, glob, json, warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pywt

from scipy.signal  import butter, filtfilt, welch
from sklearn.model_selection import train_test_split
from matplotlib.patches import Patch

np.random.seed(42)
print("✅ All imports OK")

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Drive mounted at /content/drive")

## 4 · Configuration

In [ ]:

EEG_RAW_DIR = '/content/drive/MyDrive/Depression_FL_Project/data/raw/MODMA'
META_PATH   = '/content/drive/MyDrive/Depression_FL_Project/data/raw/MODMA/subjects_information_EEG_3channels_resting_lanzhou_2015.xlsx'
OUTPUT_DIR  = '/content/drive/MyDrive/Depression_FL_Project/data/processed/eeg'
# ════════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Signal parameters ────────────────────────────────────────────
FS            = 250        # Hz
CHANNELS      = ['Fp1', 'Fpz', 'Fp2']
N_CHANNELS    = 3

# ── Filter ───────────────────────────────────────────────────────
LOWCUT        = 1.0        # Hz  — removes baseline drift
HIGHCUT       = 45.0       # Hz  — below 50 Hz power-line noise
FILTER_ORDER  = 4

# ── Epoch ────────────────────────────────────────────────────────
EPOCH_SEC     = 4
OVERLAP       = 0.5
EPOCH_SAMPLES = EPOCH_SEC * FS                      # 1000 samples
STEP_SAMPLES  = int(EPOCH_SAMPLES * (1 - OVERLAP))  # 500 samples

# ── Max recording to use per subject ────────────────────────────
# Shi et al. (2021) used 40 s; we use 80 s for more epochs
MAX_SECONDS   = 80
MAX_SAMPLES   = MAX_SECONDS * FS

# ── Train / Val / Test split (subject-level) ─────────────────────
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15

# ── Label encoding ───────────────────────────────────────────────
LABEL_MAP = {'MDD': 1, 'HC': 0}

print(f"EEG_RAW_DIR : {EEG_RAW_DIR}")
print(f"META_PATH   : {META_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print()
print(f"Epoch shape     : ({N_CHANNELS}, {EPOCH_SAMPLES})  [channels × samples]")
print(f"Epochs / subject: ~{(MAX_SAMPLES - EPOCH_SAMPLES) // STEP_SAMPLES + 1} windows from {MAX_SECONDS}s")

# ── Verify paths ─────────────────────────────────────────────────
for label, path in [("EEG folder", EEG_RAW_DIR), ("Excel file", META_PATH)]:
    exists = os.path.exists(path)
    icon   = "✅" if exists else "❌  NOT FOUND — check path"
    print(f"{icon}  {label}: {path}")

## 5 · Load subject metadata

In [ ]:
def load_metadata(meta_path):
    df = pd.read_excel(meta_path, sheet_name='Sheet1')
    df = df.iloc[:, :11]
    df.columns = ['subject_id', 'type', 'age', 'gender', 'education_years',
                  'PHQ9', 'CTQ_SF', 'LES', 'SSRS', 'GAD7', 'PSQI']
    df = df.dropna(subset=['subject_id'])
    df['subject_id'] = df['subject_id'].astype(str).str.strip()
    df['label']      = df['type'].map(LABEL_MAP)
    return df

meta_df = load_metadata(META_PATH)

print(f"Total subjects : {len(meta_df)}")
print(meta_df['type'].value_counts().to_string())
print()
print(meta_df[['subject_id','type','age','gender','PHQ9','label']].to_string(index=False))

### Demographics overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for grp, col in [('MDD','#E74C3C'), ('HC','#2ECC71')]:
    s = meta_df[meta_df['type'] == grp]
    axes[0].hist(s['age'], bins=12, alpha=0.6, color=col, label=grp, edgecolor='white')
axes[0].set_title('Age Distribution', fontsize=13)
axes[0].set_xlabel('Age'); axes[0].legend()

meta_df.groupby(['type','gender']).size().unstack().plot(
    kind='bar', ax=axes[1], color=['#F39C12','#3498DB'], edgecolor='white')
axes[1].set_title('Gender by Group', fontsize=13)
axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=0)

for grp, col in [('MDD','#E74C3C'), ('HC','#2ECC71')]:
    s = meta_df[meta_df['type'] == grp]
    axes[2].hist(s['PHQ9'], bins=12, alpha=0.6, color=col, label=grp, edgecolor='white')
axes[2].set_title('PHQ-9 Score Distribution', fontsize=13)
axes[2].set_xlabel('PHQ-9 score'); axes[2].legend()

plt.suptitle('MODMA 3-Channel EEG — Subject Demographics', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'demographics.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved")

## 6 · Subject-wise train / val / test split

In [ ]:
def subject_wise_split(df, train_r=0.70, val_r=0.15, test_r=0.15, seed=42):
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6

    def split_group(g):
        tr, tmp = train_test_split(g, test_size=1-train_r, random_state=seed)
        v_size  = val_r / (val_r + test_r)
        v, te   = train_test_split(tmp, test_size=1-v_size, random_state=seed)
        return tr, v, te

    mdd_tr, mdd_v, mdd_te = split_group(df[df['type']=='MDD'])
    hc_tr,  hc_v,  hc_te  = split_group(df[df['type']=='HC'])

    def tag(d, s):
        d = d.copy(); d['split'] = s; return d

    train_m = pd.concat([tag(mdd_tr,'train'), tag(hc_tr,'train')]).sample(frac=1, random_state=seed)
    val_m   = pd.concat([tag(mdd_v, 'val'),   tag(hc_v, 'val')  ]).sample(frac=1, random_state=seed)
    test_m  = pd.concat([tag(mdd_te,'test'),  tag(hc_te,'test') ]).sample(frac=1, random_state=seed)
    return train_m, val_m, test_m

train_meta, val_meta, test_meta = subject_wise_split(meta_df)
all_split_meta = pd.concat([train_meta, val_meta, test_meta])
all_split_meta.to_csv(os.path.join(OUTPUT_DIR, 'subject_splits.csv'), index=False)

print(f"{'Split':<8} {'Total':>6}  {'MDD':>5}  {'HC':>5}")
print("-" * 28)
for name, df in [('TRAIN', train_meta), ('VAL', val_meta), ('TEST', test_meta)]:
    c = df['type'].value_counts()
    print(f"{name:<8} {len(df):>6}  {c.get('MDD',0):>5}  {c.get('HC',0):>5}")

print()
print("✅ subject_splits.csv saved")

## 7 · EEG file loading

In [ ]:
def find_eeg_file(raw_dir, subject_id):
    base_ids = [
        subject_id,
        subject_id.lstrip('0'),
        subject_id.zfill(8),
    ]
    suffixes = ['_still', '_rest', '_resting', '']
    for bid in base_ids:
        for suf in suffixes:
            for ext in ['.txt', '.TXT']:
                for pat in [
                    os.path.join(raw_dir, f'{bid}{suf}{ext}'),
                    os.path.join(raw_dir, '**', f'{bid}{suf}{ext}'),
                ]:
                    hits = glob.glob(pat, recursive=True)
                    if hits:
                        return hits[0]
    return None

def load_eeg_txt(filepath, max_samples=None):
    """
    Load MODMA 3-ch EEG from TXT.

    Raw format: 8 columns in hex.  Columns 0,1,2 = Fp1, Fpz, Fp2.
    Returns: np.ndarray  shape (3, n_samples)  float32
    """
    try:
        raw = pd.read_csv(filepath, header=None, sep=r'\s+', dtype=str)
    except Exception:
        raw = pd.read_csv(filepath, header=None, dtype=str)

    if raw.shape[1] < 3:
        raise ValueError(f"Expected ≥3 columns, got {raw.shape[1]}")

    def hex_col(series):
        try:
            return series.apply(lambda x: int(str(x).strip(), 16)).astype(np.float32)
        except ValueError:
            return pd.to_numeric(series, errors='coerce').astype(np.float32)

    arrays = [hex_col(raw.iloc[:, i].dropna()).values for i in range(3)]
    min_len = min(len(a) for a in arrays)
    eeg = np.stack([a[:min_len] for a in arrays], axis=0)  # (3, n_samples)

    if max_samples and eeg.shape[1] > max_samples:
        eeg = eeg[:, :max_samples]
    return eeg


# ── Quick smoke test ─────────────────────────────────────────────
example_id   = meta_df['subject_id'].iloc[0]
example_file = find_eeg_file(EEG_RAW_DIR, example_id)

if example_file:
    tmp = load_eeg_txt(example_file, max_samples=MAX_SAMPLES)
    print(f"✅ Loaded {example_id}: shape={tmp.shape}  duration={tmp.shape[1]/FS:.1f}s")
    print(f"   Value range: [{tmp.min():.1f}, {tmp.max():.1f}]")
else:
    print(f"❌ File not found for {example_id}")
    print(f"   Check that TXT files are in: {EEG_RAW_DIR}")

## 8 · Signal processing functions

In [ ]:
# ── 8a. DC removal ────────────────────────────────────────────────
def remove_dc(eeg):
    return (eeg - eeg.mean(axis=1, keepdims=True)).astype(np.float32)


# ── 8b. Butterworth bandpass ──────────────────────────────────────
def bandpass_filter(eeg, lowcut=LOWCUT, highcut=HIGHCUT, fs=FS, order=FILTER_ORDER):
    """Zero-phase bandpass filter. eeg: (ch, samples) → same shape."""
    nyq  = fs / 2.0
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, eeg, axis=1).astype(np.float32)


# ── 8c. Wavelet denoising (eye-blink artifact removal) ────────────
def wavelet_denoise(eeg, wavelet='db4', level=4, threshold_factor=3.0):
    """
    Per-channel soft-threshold wavelet denoising.
    Effective for frontal eye-blink artifacts on Fp1/Fpz/Fp2.
    """
    cleaned = np.zeros_like(eeg)
    for ch in range(eeg.shape[0]):
        coeffs  = pywt.wavedec(eeg[ch], wavelet, level=level)
        sigma   = np.median(np.abs(coeffs[-1])) / 0.6745
        thresh  = threshold_factor * sigma * np.sqrt(2 * np.log(len(eeg[ch])))
        new_c   = [coeffs[0]] + [pywt.threshold(c, thresh, mode='soft') for c in coeffs[1:]]
        cleaned[ch] = pywt.waverec(new_c, wavelet)[:eeg.shape[1]]
    return cleaned.astype(np.float32)


# ── 8d. Full pipeline for one subject ────────────────────────────
def preprocess_subject(eeg_raw):
    eeg = remove_dc(eeg_raw)
    eeg = bandpass_filter(eeg)
    eeg = wavelet_denoise(eeg)
    return eeg


# ── 8e. Amplitude-based epoch rejection ──────────────────────────
def reject_bad_epochs(epochs, labels, amp_threshold=None):
    peak_amps = np.max(np.abs(epochs), axis=(1, 2))

    if amp_threshold is None:
        threshold = peak_amps.mean() + 5.0 * peak_amps.std()
    else:
        threshold = amp_threshold

    keep     = peak_amps < threshold
    rej_rate = 1.0 - keep.mean()
    return epochs[keep], labels[keep], rej_rate

# ── 8f. Epoch segmentation ───────────────────────────────────────
def epoch_signal(eeg, epoch_samples=EPOCH_SAMPLES, step_samples=STEP_SAMPLES):
    """
    Segment continuous EEG into overlapping windows.
    Returns (n_epochs, n_channels, epoch_samples)  float32
    """
    n   = eeg.shape[1]
    starts = range(0, n - epoch_samples + 1, step_samples)
    return np.stack([eeg[:, s:s+epoch_samples] for s in starts]).astype(np.float32)


print("✅ Signal processing functions ready")

### Visualize preprocessing effect (first subject)

In [ ]:
if example_file:
    eeg_raw  = load_eeg_txt(example_file, max_samples=MAX_SAMPLES)
    eeg_proc = preprocess_subject(eeg_raw)

    t      = np.arange(eeg_raw.shape[1]) / FS
    n_plot = 10 * FS  # plot 10 seconds

    fig, axes = plt.subplots(2, 3, figsize=(17, 6))
    for ci, ch in enumerate(CHANNELS):
        axes[0, ci].plot(t[:n_plot], eeg_raw[ci, :n_plot],
                          color='#3498DB', lw=0.6)
        axes[0, ci].set_title(f'{ch} — Raw', fontsize=12)
        axes[0, ci].set_ylabel('ADC units')

        axes[1, ci].plot(t[:n_plot], eeg_proc[ci, :n_plot],
                          color='#E74C3C', lw=0.6)
        axes[1, ci].set_title(f'{ch} — Preprocessed', fontsize=12)
        axes[1, ci].set_ylabel('µV (normalised)'); axes[1, ci].set_xlabel('Time (s)')

    sid_type = meta_df[meta_df.subject_id == example_id]['type'].values[0]
    plt.suptitle(f'Subject {example_id} ({sid_type}) — First 10 s', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'preprocessing_example.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved preprocessing_example.png")
else:
    print("⚠️  No file found — set EEG_RAW_DIR and re-run from cell 7.")

## 9 · Normalization helpers

In [ ]:
def compute_norm_stats(epochs_list):
    """
    Compute per-channel mean & std from a list of epoch arrays.
    epochs_list: list of (n_epochs, n_ch, n_samples)
    Returns: mean (n_ch,), std (n_ch,)
    """
    all_data = np.concatenate(epochs_list, axis=0)
    mean = all_data.mean(axis=(0, 2))
    std  = all_data.std(axis=(0, 2))
    std  = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def normalize_epochs(epochs, mean, std):
    """Z-score per channel. epochs: (n, ch, t), mean/std: (ch,)"""
    return ((epochs - mean[None, :, None]) / std[None, :, None]).astype(np.float32)


print("✅ Normalization helpers ready")

## 10 · Main processing loop — all subjects

In [ ]:
def process_all_subjects(meta_df, raw_dir,
                          max_samples=MAX_SAMPLES, amp_threshold=150.0):
    results, missing = {}, []

    for _, row in meta_df.iterrows():
        sid   = str(row['subject_id'])
        label = int(row['label'])
        split = row.get('split', 'unknown')

        fpath = find_eeg_file(raw_dir, sid)
        if fpath is None:
            missing.append(sid)
            print(f"  ❌ MISSING  {sid}")
            continue

        try:
            eeg_raw      = load_eeg_txt(fpath, max_samples=max_samples)
            eeg_proc     = preprocess_subject(eeg_raw)
            epochs       = epoch_signal(eeg_proc)
            labels_arr   = np.full(len(epochs), label, dtype=np.int64)
            ep_c, lb_c, rej = reject_bad_epochs(epochs, labels_arr, amp_threshold=None)

            results[sid] = dict(
                epochs=ep_c, labels=lb_c,
                label=label, split=split, type=row['type'],
                n_epochs=len(ep_c),
                n_rejected=len(epochs) - len(ep_c),
                rej_rate=rej,
            )
            print(f"  ✅ {sid} ({row['type']:<3})  {split:<5}  "
                  f"epochs={len(ep_c):>3}  rejected={len(epochs)-len(ep_c)}"
                  f"  ({rej*100:.0f}%)")

        except Exception as e:
            missing.append(sid)
            print(f"  ❌ ERROR  {sid}: {e}")

    return results, missing


print("▶ Processing all subjects …\n")
results, missing_subs = process_all_subjects(all_split_meta, EEG_RAW_DIR)

print(f"\n{'─'*45}")
print(f"Processed : {len(results)} / {len(all_split_meta)} subjects")
print(f"Missing   : {len(missing_subs)}")
if missing_subs:
    print("  IDs:", missing_subs)

## 11 · Assemble split arrays & normalize

In [ ]:
def assemble_split(results, split_name):
    X_list, y_list, sid_list = [], [], []
    for sid, d in results.items():
        if d['split'] == split_name and d['n_epochs'] > 0:
            X_list.append(d['epochs'])
            y_list.append(d['labels'])
            sid_list.append(np.array([sid] * d['n_epochs']))
    if not X_list:
        return None, None, None
    return (np.concatenate(X_list), np.concatenate(y_list),
            np.concatenate(sid_list))


X_train, y_train, ids_train = assemble_split(results, 'train')
X_val,   y_val,   ids_val   = assemble_split(results, 'val')
X_test,  y_test,  ids_test  = assemble_split(results, 'test')

# Normalise with training stats only
if X_train is not None:
    norm_mean, norm_std = compute_norm_stats([X_train])
    X_train = normalize_epochs(X_train, norm_mean, norm_std)
    if X_val  is not None: X_val  = normalize_epochs(X_val,  norm_mean, norm_std)
    if X_test is not None: X_test = normalize_epochs(X_test, norm_mean, norm_std)

    print("Shapes after normalisation:")
    print(f"  X_train : {X_train.shape}   y_train : {y_train.shape}")
    if X_val  is not None: print(f"  X_val   : {X_val.shape}   y_val   : {y_val.shape}")
    if X_test is not None: print(f"  X_test  : {X_test.shape}   y_test  : {y_test.shape}")

    print("\nClass balance (window-level):")
    print(f"{'Split':<8} {'MDD':>6} {'HC':>6} {'MDD%':>7}")
    for name, y_ in [('train',y_train),('val',y_val),('test',y_test)]:
        if y_ is not None:
            m = (y_==1).sum(); h = (y_==0).sum()
            print(f"  {name:<6} {m:>6} {h:>6} {m/len(y_)*100:>6.1f}%")
else:
    print("⚠️  No results — ensure EEG_RAW_DIR is correct and re-run from cell 10.")

## 12 · Save processed arrays to Drive

In [ ]:
# ── Save UN-normalized epochs FIRST ──────────────────────────
X_train_raw, y_train_raw, ids_train_raw = assemble_split(results, 'train')
X_val_raw,   y_val_raw,   ids_val_raw   = assemble_split(results, 'val')
X_test_raw,  y_test_raw,  ids_test_raw  = assemble_split(results, 'test')

np.save(os.path.join(OUTPUT_DIR, 'X_train_raw.npy'),   X_train_raw)
np.save(os.path.join(OUTPUT_DIR, 'X_val_raw.npy'),     X_val_raw)
np.save(os.path.join(OUTPUT_DIR, 'X_test_raw.npy'),    X_test_raw)
np.save(os.path.join(OUTPUT_DIR, 'y_train_raw.npy'),   y_train_raw)
np.save(os.path.join(OUTPUT_DIR, 'y_val_raw.npy'),     y_val_raw)
np.save(os.path.join(OUTPUT_DIR, 'y_test_raw.npy'),    y_test_raw)
np.save(os.path.join(OUTPUT_DIR, 'ids_train_raw.npy'), ids_train_raw.astype(str))
np.save(os.path.join(OUTPUT_DIR, 'ids_val_raw.npy'),   ids_val_raw.astype(str))
np.save(os.path.join(OUTPUT_DIR, 'ids_test_raw.npy'),  ids_test_raw.astype(str))

print(f"✅ Un-normalized epochs saved")
print(f"  X_train_raw : {X_train_raw.shape}")
print(f"  Value range : [{X_train_raw.min():.4f}, {X_train_raw.max():.4f}]")

# ── Then the existing normalized saves continue below ────────
if X_train is not None:
    splits = {
        'train': (X_train, y_train, ids_train),
        'val'  : (X_val,   y_val,   ids_val),
        'test' : (X_test,  y_test,  ids_test),
    }
    for split_name, (X, y, ids) in splits.items():
        if X is not None:
            np.save(os.path.join(OUTPUT_DIR, f'X_{split_name}.npy'), X)
            np.save(os.path.join(OUTPUT_DIR, f'y_{split_name}.npy'), y)
            np.save(os.path.join(OUTPUT_DIR, f'ids_{split_name}.npy'), ids.astype(str))

    np.save(os.path.join(OUTPUT_DIR, 'norm_mean.npy'), norm_mean)
    np.save(os.path.join(OUTPUT_DIR, 'norm_std.npy'),  norm_std)

    epoch_counts = {
        sid: {'n_epochs': d['n_epochs'], 'n_rejected': d['n_rejected'],
              'label': d['label'], 'type': d['type'], 'split': d['split']}
        for sid, d in results.items()
    }
    with open(os.path.join(OUTPUT_DIR, 'epoch_counts.json'), 'w') as f:
        json.dump(epoch_counts, f, indent=2)

    print(f"✅ Saved to {OUTPUT_DIR}")
    print()
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        sz = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
        print(f"  {fname:<35}  {sz:>8.1f} KB")
else:
    print("⚠️  Nothing to save — process subjects first (cell 10).")

## 13 · Quality checks & visualizations

In [ ]:
# ── 13a. Epoch counts per subject ───────────────────────────────
if results:
    summary = pd.DataFrame([
        {'subject_id': sid, 'type': d['type'], 'split': d['split'],
         'n_epochs': d['n_epochs'], 'n_rejected': d['n_rejected'],
         'rej_pct': d['rej_rate'] * 100}
        for sid, d in results.items()
    ])

    print("Epoch count stats by split & type:")
    print(summary.groupby(['split','type'])['n_epochs'].describe().round(1).to_string())
    print(f"\nOverall rejection rate: {summary['rej_pct'].mean():.1f}%")

    fig, ax = plt.subplots(figsize=(14, 4))
    colors = summary['type'].map({'MDD':'#E74C3C', 'HC':'#2ECC71'})
    ax.bar(range(len(summary)), summary['n_epochs'], color=colors, alpha=0.85, edgecolor='white')
    ax.set_xlabel('Subject (sorted by processing order)', fontsize=11)
    ax.set_ylabel('Epochs retained', fontsize=11)
    ax.set_title('Epochs per Subject After Artifact Rejection', fontsize=13)
    ax.legend(handles=[Patch(color='#E74C3C', label='MDD'),
                        Patch(color='#2ECC71', label='HC')], fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'epoch_counts.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 13b. PSD comparison: MDD vs HC ──────────────────────────────
if X_train is not None:
    BANDS = [('δ',1,4), ('θ',4,8), ('α',8,13), ('β',13,30), ('γ',30,45)]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ci, ch in enumerate(CHANNELS):
        for lv, ln, col in [(1,'MDD','#E74C3C'), (0,'HC','#2ECC71')]:
            mask = y_train == lv
            psds = []
            for ep in X_train[mask, ci, :]:
                f, p = welch(ep, fs=FS, nperseg=256)
                psds.append(p)
            mean_p = np.mean(psds, axis=0)
            std_p  = np.std(psds,  axis=0)
            fm     = f <= 45

            axes[ci].semilogy(f[fm], mean_p[fm], color=col, label=ln, lw=2)
            axes[ci].fill_between(f[fm],
                                   np.clip(mean_p-std_p, 1e-10, None)[fm],
                                   (mean_p+std_p)[fm],
                                   color=col, alpha=0.12)

        for bname, blo, bhi in BANDS:
            axes[ci].axvspan(blo, bhi, alpha=0.04, color='gray')
            axes[ci].text((blo+bhi)/2, axes[ci].get_ylim()[0]*1.5,
                           bname, ha='center', fontsize=9, color='gray')

        axes[ci].set_title(f'{ch} — Power Spectral Density', fontsize=12)
        axes[ci].set_xlabel('Frequency (Hz)')
        if ci == 0: axes[ci].set_ylabel('Power (V²/Hz)')
        axes[ci].legend(); axes[ci].grid(True, alpha=0.25)

    plt.suptitle('PSD: MDD vs HC  (training set)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'psd_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved psd_comparison.png")

In [ ]:
# ── 13c. Sample epoch plot (one MDD, one HC) ────────────────────
if X_train is not None:
    t_ep = np.arange(EPOCH_SAMPLES) / FS
    fig, axes = plt.subplots(2, 3, figsize=(17, 6))

    for ri, (lv, ln) in enumerate([(1,'MDD'), (0,'HC')]):
        idx = np.where(y_train == lv)[0][0]
        ep  = X_train[idx]
        col = '#E74C3C' if lv == 1 else '#2ECC71'
        for ci, ch in enumerate(CHANNELS):
            axes[ri, ci].plot(t_ep, ep[ci], color=col, lw=0.9)
            axes[ri, ci].set_title(f'{ln} — {ch}', fontsize=12)
            axes[ri, ci].set_xlabel('Time (s)')
            if ci == 0: axes[ri, ci].set_ylabel('Normalised amplitude')
            axes[ri, ci].grid(True, alpha=0.25)

    plt.suptitle('Sample 4-second Epochs After Full Preprocessing', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'sample_epochs.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved sample_epochs.png")

## 14 · Summary & next steps

In [ ]:
print("=" * 60)
print("  MODMA EEG Preprocessing — Complete")
print("=" * 60)
print(f"  Sampling rate      : {FS} Hz")
print(f"  Channels           : {CHANNELS}")
print(f"  Bandpass           : {LOWCUT}–{HIGHCUT} Hz  (Butterworth order {FILTER_ORDER})")
print(f"  Artifact removal   : Wavelet soft-threshold  (db4, level 4)")
print(f"  Epoch length       : {EPOCH_SEC}s  ({EPOCH_SAMPLES} samples)")
print(f"  Epoch overlap      : {int(OVERLAP*100)}%  ({STEP_SAMPLES}-sample hop)")
print(f"  Max recording used : {MAX_SECONDS}s per subject")
print(f"  Normalisation      : Z-score per channel (train stats only)")
print(f"  Split method       : Subject-wise stratified — NO window leakage")
print()
if X_train is not None:
    print(f"  Subjects processed : {len(results)}")
    print(f"  Train epochs       : {len(X_train)}  shape {X_train.shape}")
    if X_val  is not None: print(f"  Val   epochs       : {len(X_val)}")
    if X_test is not None: print(f"  Test  epochs       : {len(X_test)}")
print()
print("  Output files (all in OUTPUT_DIR):")
print("    X_train/val/test.npy   ← model-ready  (n, 3, 1000)")
print("    y_train/val/test.npy   ← binary labels (1=MDD, 0=HC)")
print("    ids_*.npy              ← subject ID per epoch")
print("    norm_mean/std.npy      ← normalisation stats")
print("    subject_splits.csv     ← which subject → which split")
print("    epoch_counts.json      ← per-subject counts (for FL weighting)")
print()
print("  Next notebooks:")
print("    02_EEG_FeatureExtraction.ipynb  ← DE, power, nonlinear features")
print("    03_EEG_DeepModel.ipynb          ← 1D-CNN + Bi-LSTM + attention")
print("=" * 60)

## 15 · Quick-load helper

In [ ]:
def load_processed_eeg(output_dir=OUTPUT_DIR):
    """Reload all preprocessed EEG arrays saved by this notebook."""
    data = {}
    for split in ['train', 'val', 'test']:
        for kind in ['X', 'y', 'ids']:
            fp = os.path.join(output_dir, f'{kind}_{split}.npy')
            if os.path.exists(fp):
                data[f'{kind}_{split}'] = np.load(fp, allow_pickle=True)
    for stat in ['norm_mean', 'norm_std']:
        fp = os.path.join(output_dir, f'{stat}.npy')
        if os.path.exists(fp):
            data[stat] = np.load(fp)
    return data

# Verify
loaded = load_processed_eeg()
print("Arrays loaded from Drive:")
for k, v in loaded.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:<20} {v.shape}")